In [ ]:
import sqlite3
import re

# 创建内存数据库
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 读取原始 SQL 文件内容
sql_file_path = "../query_dataset/business_descriptions.sql"
with open(sql_file_path, "r", encoding="utf-8") as f:
    sql_script = f.read()

# ✅ 移除所有 COLLATE utf8mb4_unicode_ci 子句
sql_script = re.sub(r"COLLATE\s+utf8mb4_unicode_ci", "", sql_script, flags=re.IGNORECASE)

try:
    cursor.executescript(sql_script)
    print("✅ SQL 脚本已成功执行（已移除 COLLATE）")
except Exception as e:
    print("❌ 执行 SQL 脚本失败：", e)


In [ ]:
import mysql.connector

# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",  # 替换为你自己的密码
    database="ucb_db"
)

cursor = conn.cursor()

# 查询所有描述
cursor.execute("SELECT description FROM business_descriptions")
descriptions = [row[0] for row in cursor.fetchall() if row[0]]

# 筛选包含 Los Angeles 的描述
la_descriptions = [desc for desc in descriptions if "Los Angeles" in desc]

print(f"✅ 共找到 {len(la_descriptions)} 条包含 'Los Angeles' 的描述")
for desc in la_descriptions[:5]:
    print("📝", desc)


In [ ]:
import mysql.connector
import openai
import time
import json
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看


# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",
    database="ucb_db"
)
cursor = conn.cursor()

# 获取所有字段（包括 description）
cursor.execute("SELECT * FROM business_descriptions")
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

# 定义 GPT 判断函数
def is_los_angeles(description, client, deployment_name):
    # 构造 prompt
    prompt = (
        "Does the following business description mention that it is located in Los Angeles?\n"
        "Just answer 'yes' or 'no'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts geographic location from business descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower().startswith("yes")

    except Exception as e:
        print(f"❌ Error determining location from description: {e}")
        return False

# ✅ 用于保存符合条件的记录
la_data = []

# ✅ 遍历记录进行 GPT 判断
for i, row in enumerate(rows[:100]):  # 可根据需要扩大范围
    record = dict(zip(column_names, row))
    description = record.get("description", "")

    if description and is_los_angeles(description, client, deployment_name):
        la_data.append(record)
        print(f"✅ [{i}] 是洛杉矶: {description}")
    else:
        print(f"❌ [{i}] 不是洛杉矶")

print(f"\n🎯 最终保留 {len(la_data)} 条位于洛杉矶的完整记录")


In [ ]:
la_data

In [1]:
import sqlite3

# 尝试连接 SQLite 数据库
db_path = "../query_dataset/review_query.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 查看数据库中所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("📋 数据库中包含的表：", tables)


📋 数据库中包含的表： [('review',)]


In [2]:
cursor.execute("SELECT * FROM review LIMIT 5;")
rows = cursor.fetchall()
colnames = [desc[0] for desc in cursor.description]

print("字段名：", colnames)
for row in rows:
    print(row)


字段名： ['name', 'time', 'rating', 'text', 'gmap_id']
('Michael Rizal', 'September 03, 2020 at 04:15 PM', 5, 'Located in the vibrant area of Los Angeles, CA 90023, this company truly stands out. "Great company. Amazing customer service and they always have what we need in stock. Sometimes, we’d ask to hold for future orders and they will! Miss Jane is very helpful and great communicator."', 'gmap_0')
('Faranak Rafizadeh', '2021-04-12 17:07:52', 5, 'Los Angeles is known for its vibrant culture and friendly atmosphere. "Nice people helpful."', 'gmap_0')
('Javier Perez', '2018-04-23 16:24:26', 5, 'I had a fantastic experience at this amazing spot in Los Angeles, CA 90023, where the friendly staff went above and beyond to make my visit truly enjoyable!', 'gmap_0')
('Luis P.', '2017-07-10 22:12:19', 5, 'I had an amazing experience at this charming café in Los Angeles, where the friendly staff and delicious pastries made my day truly special!', 'gmap_0')
('His Mama Cakez', 'May 19, 2021 at 03:5